In [ ]:
!pip install datasets

In [ ]:
import pandas as pd

data = {
    "text": [
        "I love this movie",
        "This product is amazing",
        "Fantastic customer service",
        "The food was delicious",
        "I am very happy with the purchase",
        "Great quality and fast delivery",
        "Excellent experience overall",
        "This app is very useful",
        "Highly recommended",
        "The book was interesting",
        "I enjoyed every moment",
        "Amazing performance",
        "The staff was friendly",
        "Best purchase ever",
        "Wonderful experience",
        "I am satisfied",
        "Very good product",
        "Works perfectly",
        "Outstanding quality",
        "Loved it",
        "I hate this service",
        "Very bad experience",
        "The product is terrible",
        "Waste of money",
        "Not worth buying",
        "Poor customer support",
        "I am disappointed",
        "Worst experience ever",
        "The app keeps crashing",
        "The food was awful",
        "Bad quality",
        "Completely useless",
        "I regret buying this",
        "Very frustrating",
        "The book was boring",
        "The staff was rude",
        "Delivery was late",
        "Not satisfied",
        "Terrible performance",
        "It stopped working",
        "Good value for money",
        "Happy with the service",
        "The design is beautiful",
        "Very reliable product",
        "The camera quality is excellent",
        "Poor battery life",
        "The screen is broken",
        "Customer service was unhelpful",
        "The software is buggy",
        "Excellent battery backup"
    ],
    "label": [
        1,1,1,1,1,1,1,1,1,1,
        1,1,1,1,1,1,1,1,1,1,
        0,0,0,0,0,0,0,0,0,0,
        0,0,0,0,0,0,0,0,0,0,
        1,1,1,1,1,
        0,0,0,0,
        1
    ]
}

df = pd.DataFrame(data)
df.head()

,text,label
0,I love this movie,1
1,This product is amazing,1
2,Fantastic customer service,1
3,The food was delicious,1
4,I am very happy with the purchase,1


In [ ]:
df.to_csv("sample_data.csv", index=False)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="sample_data.csv")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [ ]:
from sklearn.model_selection import train_test_split

dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 32
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 8
    })
})

In [ ]:
print((dataset["train"]))
# final traning detaset

Dataset({
    features: ['text', 'label'],
    num_rows: 32
})


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenise(example):
  return tokenizer(
      example["text"],
      padding="max_length",
      truncation=True,
  )

tokenized_data = dataset.map(tokenise)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"]
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50, training_loss=0.27911062240600587, metrics={'train_runtime': 53.875, 'train_samples_per_second': 7.425, 'train_steps_per_second': 0.928, 'total_flos': 105244422144000.0, 'train_loss': 0.27911062240600587, 'epoch': 10.0})

In [ ]:
model.save_pretrained("sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./results/checkpoint-50"
)

classifier("This product is fantastic")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.7013164758682251}]

In [ ]:
classifier("I hate this product")

[{'label': 'LABEL_0', 'score': 0.7013164758682251}]

In [ ]:
classifier("Amazing quality")

[{'label': 'LABEL_0', 'score': 0.7054241299629211}]

In [ ]:
classifier("Worst purchase ever")

[{'label': 'LABEL_0', 'score': 0.7053031921386719}]